In [1]:
!pip install transformers accelerate datasets pandas

In [2]:
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import random
import time
import re
import pandas as pd

In [3]:
SEED = 42
random.seed(SEED)

In [4]:
subjects = ["algebra", "geometry", "number_theory", "prealgebra"]

datasets_list = []

for sub in subjects:
    print(f"Loading {sub}...")
    ds_part = load_dataset("EleutherAI/hendrycks_math", sub, split="train")
    datasets_list.append(ds_part)

ds = concatenate_datasets(datasets_list)

print("Total samples:", len(ds))

Loading algebra...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

...

prealgebra/train-00000-of-00001.parquet:   0%|          | 0.00/384k [00:00<?, ?B/s]

prealgebra/test-00000-of-00001.parquet:   0%|          | 0.00/268k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1205 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/871 [00:00<?, ? examples/s]

Total samples: 4688


In [5]:
easy, medium, hard = [], [], []

for i, row in enumerate(ds):
    level = row['level']

    if "1" in level:
        easy.append(i)
    elif "3" in level:
        medium.append(i)
    elif "5" in level:
        hard.append(i)

random.shuffle(easy)
random.shuffle(medium)
random.shuffle(hard)

sampled = easy[:20] + medium[:20] + hard[:10]

problems = [ds[i] for i in sampled]

print("Sampled problems:", len(problems))

Sampled problems: 50


In [6]:
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("BF16 model loaded!")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

BF16 model loaded!


In [7]:
def solve(problem):
    prompt = f"Question: {problem['problem']}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
def extract_number(text):
    nums = re.findall(r'-?\d+', text)
    return int(nums[-1]) if nums else None

In [9]:
print("="*60)
print("Running BF16 on 50 samples")
print("="*60)

bf16_results = []
start_time = time.time()

for i, p in enumerate(problems):
    output = solve(p)

    pred = extract_number(output)
    gt = extract_number(p["solution"])

    correct = (pred == gt)

    elapsed = int((time.time() - start_time) / 60)

    print(f"[{i+1}/50] diff={p['level']} correct={correct} elapsed={elapsed}min")

    bf16_results.append({
        "pid": i,
        "difficulty": p["level"],
        "prediction": pred,
        "ground_truth": gt,
        "correct": correct,
        "model_output": output
    })

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Running BF16 on 50 samples


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[1/50] diff=Level 1 correct=False elapsed=0min


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

[47/50] diff=Level 5 correct=False elapsed=5min


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[48/50] diff=Level 5 correct=True elapsed=5min


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[49/50] diff=Level 5 correct=False elapsed=5min
[50/50] diff=Level 5 correct=True elapsed=5min


In [10]:
df = pd.DataFrame(bf16_results)

print("Total rows:", len(df))   # MUST BE 50
print(df.head())

df.to_csv("qwen2_math_bf16_50.csv", index=False)

print("CSV saved!")

from google.colab import files
files.download("qwen2_math_bf16_50.csv")

Total rows: 50
   pid difficulty  prediction  ground_truth  correct  \
0    0    Level 1           8            38    False   
1    1    Level 1           2             2     True   
2    2    Level 1           3             3     True   
3    3    Level 1         136            11    False   
4    4    Level 1           2             2     True   

                                        model_output  
0  Question: Suppose that we have an 8-sided die ...  
1  Question: Compute $(-64)\div (-32)$.\nAnswer: ...  
2  Question: When rolling a fair 6-sided die, wha...  
3  Question: When Joyce counts the pennies in her...  
4  Question: If $x+y = 6$ and $x^2-y^2 = 12$, the...  
CSV saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>